# 🧮 AIMO Problem Space Explorer (GPU-Accelerated)

Visualize AIMO 1, 2, and 3 competition problem embeddings using **RAPIDS cuML** for GPU-accelerated UMAP.

## 📊 Features
- ✅ **GPU-Accelerated**: RAPIDS cuML UMAP (100x faster than CPU)
- ✅ **CPU Fallback**: Works without RAPIDS using umap-learn
- ✅ **2D & 3D Visualizations**: Interactive Plotly scatter plots
- ✅ **Color Coding**: By AIMO version, answer status, and source
- ✅ **Adaptive Rendering**: Automatic opacity/size based on data density
- ✅ **Export**: Save to interactive HTML files
- ✅ **White Theme**: Clean white background with LaTeX math rendering
- ✅ **LaTeX Support**: MathJax-enabled for rendering $\LaTeX$ formulas in hover text


In [1]:
# Check for RAPIDS availability
try:
    import cudf
    import cuml
    import cupy as cp
    from cuml.manifold import UMAP as cumlUMAP
    RAPIDS_AVAILABLE = True
    print("✅ RAPIDS (cuDF, cuML) is available - GPU acceleration enabled!")
except ImportError:
    RAPIDS_AVAILABLE = False
    print("⚠️  RAPIDS not available - falling back to CPU (umap-learn)")
    print("   To install RAPIDS: pip install cudf-cu13 cuml-cu13 --extra-index-url=https://pypi.nvidia.com")


⚠️  RAPIDS not available - falling back to CPU (umap-learn)
   To install RAPIDS: pip install cudf-cu13 cuml-cu13 --extra-index-url=https://pypi.nvidia.com


In [2]:
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from typing import Optional, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# RAPIDS or CPU fallback for UMAP
if RAPIDS_AVAILABLE:
    from cuml.manifold import UMAP as cumlUMAP
    print("✅ Using cuML UMAP (GPU)")
else:
    import umap as cpuUMAP
    print("✅ Using umap-learn (CPU)")

# Configure Plotly with MathJax/LaTeX support
import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Enable MathJax for LaTeX rendering in Plotly
# LaTeX expressions in hover text using $...$ will be rendered
pio.templates["plotly_white"].layout.update(
    font=dict(family="Computer Modern, serif", size=12),
)

print(f"\n🖥️  Backend: {'GPU (RAPIDS cuML)' if RAPIDS_AVAILABLE else 'CPU (umap-learn)'}")
print("📐 LaTeX rendering enabled via MathJax")


✅ Using umap-learn (CPU)

🖥️  Backend: CPU (umap-learn)
📐 LaTeX rendering enabled via MathJax


## ⚙️ Configuration


In [3]:
# =============================================================================
# PATH CONFIGURATION
# =============================================================================
EMBEDDINGS_DIR = Path("embeddings")
NUMPY_DIR = EMBEDDINGS_DIR / "numpy"
CSV_PATH = EMBEDDINGS_DIR / "aimo_embeddings.csv"
OUTPUT_DIR = Path("visualizations")
OUTPUT_DIR.mkdir(exist_ok=True)

# =============================================================================
# UMAP PARAMETERS
# =============================================================================
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_METRIC = "cosine"
RANDOM_SEED = 42

# =============================================================================
# COLOR SCHEMES
# =============================================================================
AIMO_COLORS = {
    'AIMO 1': '#FF6B6B',  # Coral red
    'AIMO 2': '#4ECDC4',  # Teal
    'AIMO 3': '#9B59B6',  # Purple
}

ANSWER_STATUS_COLORS = {
    'Has Answer': '#2ECC71',   # Green
    'Unanswered': '#E74C3C',   # Red
}

SOURCE_COLORS = {
    'aimo1_train': '#FF6B6B',
    'aimo1_test': '#FF8E8E',
    'aimo2_reference': '#4ECDC4',
    'aimo2_test': '#7EDCD6',
    'aimo3_reference': '#9B59B6',
    'aimo3_data': '#B07CC6',
    'aimo3_test': '#C59DD6',
}

TYPE_COLORS = {
    'Problem': '#3498DB',  # Blue
    'Answer': '#F39C12',   # Orange
}

print("⚙️  Configuration loaded")
print(f"   UMAP: n_neighbors={UMAP_N_NEIGHBORS}, min_dist={UMAP_MIN_DIST}, metric={UMAP_METRIC}")


⚙️  Configuration loaded
   UMAP: n_neighbors=15, min_dist=0.1, metric=cosine


## 📥 Load Embeddings


In [4]:
def load_embeddings() -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """
    Load embeddings from numpy arrays (faster) or CSV (fallback).
    
    Returns:
        df: DataFrame with metadata
        problem_embeddings: numpy array of problem embeddings
        answer_embeddings: numpy array of answer embeddings
    """
    if NUMPY_DIR.exists():
        print("📂 Loading from numpy arrays...")
        problem_embeddings = np.load(NUMPY_DIR / "problem_embeddings.npy")
        answer_embeddings = np.load(NUMPY_DIR / "answer_embeddings.npy")
        df = pd.read_csv(NUMPY_DIR / "metadata.csv")
        print(f"✅ Loaded {len(df)} records")
        return df, problem_embeddings, answer_embeddings
    
    elif CSV_PATH.exists():
        print("📂 Loading from CSV...")
        df = pd.read_csv(CSV_PATH)
        
        print("   Parsing problem embeddings...")
        problem_embeddings = np.array([json.loads(e) for e in df['problem_embedding']], dtype=np.float32)
        
        print("   Parsing answer embeddings...")
        answer_embeddings = np.array([json.loads(e) for e in df['answer_embedding']], dtype=np.float32)
        
        df = df.drop(columns=['problem_embedding', 'answer_embedding'])
        print(f"✅ Loaded {len(df)} records")
        return df, problem_embeddings, answer_embeddings
    
    else:
        raise FileNotFoundError(
            "No embeddings found. Run extract_aimo_problems.py first:\n"
            "  python extract_aimo_problems.py"
        )

def get_aimo_version(source: str) -> str:
    """Extract AIMO version from source string."""
    if 'aimo1' in source:
        return 'AIMO 1'
    elif 'aimo2' in source:
        return 'AIMO 2'
    elif 'aimo3' in source:
        return 'AIMO 3'
    return 'Unknown'

def wrap_text_for_hover(text: str, max_chars: int = 80, max_lines: int = 8) -> str:
    """
    Wrap text for Plotly hover tooltips by inserting <br> tags.
    
    Args:
        text: Input text to wrap
        max_chars: Maximum characters per line
        max_lines: Maximum number of lines before truncation
    
    Returns:
        Text with <br> tags for line breaks
    """
    if not text or len(text) == 0:
        return text
    
    text = str(text)
    words = text.split()
    lines = []
    current_line = []
    current_length = 0
    
    for word in words:
        word_len = len(word)
        if current_length + word_len + (1 if current_line else 0) <= max_chars:
            current_line.append(word)
            current_length += word_len + (1 if len(current_line) > 1 else 0)
        else:
            if current_line:
                lines.append(' '.join(current_line))
            current_line = [word]
            current_length = word_len
            
            # Check if we've reached max lines
            if len(lines) >= max_lines - 1:
                # Add remaining as truncated
                remaining = ' '.join([word] + words[words.index(word)+1:])
                if len(remaining) > max_chars:
                    lines.append(remaining[:max_chars-3] + '...')
                else:
                    lines.append(remaining)
                return '<br>'.join(lines)
    
    # Add last line
    if current_line:
        lines.append(' '.join(current_line))
    
    return '<br>'.join(lines)

# Load the data
df, problem_embeddings, answer_embeddings = load_embeddings()

# Add derived columns
df['aimo_version'] = df['source'].apply(get_aimo_version)
df['answer_status'] = df['has_answer'].apply(lambda x: 'Has Answer' if x else 'Unanswered')
# Create problem preview with line breaks for hover tooltips
df['problem_preview'] = df['problem'].apply(
    lambda x: wrap_text_for_hover(x[:400] + '...' if len(x) > 400 else x, max_chars=70, max_lines=6)
)

print(f"\n📊 Dataset Summary:")
print(f"   Total problems: {len(df)}")
print(f"   Embedding dimension: {problem_embeddings.shape[1]}")
print(f"\n   By AIMO version:")
print(df['aimo_version'].value_counts().to_string())
print(f"\n   By answer status:")
print(df['answer_status'].value_counts().to_string())


📂 Loading from numpy arrays...
✅ Loaded 44 records

📊 Dataset Summary:
   Total problems: 44
   Embedding dimension: 4096

   By AIMO version:
aimo_version
AIMO 3    18
AIMO 1    13
AIMO 2    13

   By answer status:
answer_status
Has Answer    30
Unanswered    14


## 🚀 GPU-Accelerated UMAP Dimensionality Reduction


In [5]:
def apply_umap_gpu(embeddings: np.ndarray, n_components: int = 2) -> np.ndarray:
    """Apply GPU-accelerated UMAP using cuML."""
    print(f"🚀 Applying cuML UMAP (GPU)...")
    print(f"   Input: {embeddings.shape} → {n_components}D")
    
    embeddings_gpu = cp.asarray(embeddings, dtype=cp.float32)
    
    reducer = cumlUMAP(
        n_components=n_components,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=RANDOM_SEED,
        verbose=True
    )
    
    reduced = reducer.fit_transform(embeddings_gpu)
    result = cp.asnumpy(reduced)
    
    print(f"✅ UMAP complete: {result.shape}")
    return result


def apply_umap_cpu(embeddings: np.ndarray, n_components: int = 2) -> np.ndarray:
    """Apply CPU UMAP using umap-learn."""
    print(f"🐢 Applying CPU UMAP...")
    print(f"   Input: {embeddings.shape} → {n_components}D")
    
    reducer = cpuUMAP.UMAP(
        n_components=n_components,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=RANDOM_SEED,
        verbose=True
    )
    
    result = reducer.fit_transform(embeddings)
    print(f"✅ UMAP complete: {result.shape}")
    return result


def apply_umap(embeddings: np.ndarray, n_components: int = 2) -> np.ndarray:
    """Apply UMAP using GPU if available, else CPU."""
    if RAPIDS_AVAILABLE:
        return apply_umap_gpu(embeddings, n_components)
    else:
        return apply_umap_cpu(embeddings, n_components)


In [6]:
# Apply UMAP to problem embeddings (2D and 3D)
print("=" * 70)
print("🔮 UMAP: Problem Embeddings")
print("=" * 70)

print("\n📊 2D Projection:")
problem_coords_2d = apply_umap(problem_embeddings, n_components=2)

print("\n📊 3D Projection:")
problem_coords_3d = apply_umap(problem_embeddings, n_components=3)

# Add to dataframe
df['prob_x2d'] = problem_coords_2d[:, 0]
df['prob_y2d'] = problem_coords_2d[:, 1]
df['prob_x3d'] = problem_coords_3d[:, 0]
df['prob_y3d'] = problem_coords_3d[:, 1]
df['prob_z3d'] = problem_coords_3d[:, 2]

print("\n✅ Problem coordinates added to dataframe")


🔮 UMAP: Problem Embeddings

📊 2D Projection:
🐢 Applying CPU UMAP...
   Input: (44, 4096) → 2D
UMAP(angular_rp_forest=True, metric='cosine', n_jobs=1, random_state=42, verbose=True)
Sun Dec 28 16:35:16 2025 Construct fuzzy simplicial set
Sun Dec 28 16:35:16 2025 Finding Nearest Neighbors
Sun Dec 28 16:35:20 2025 Finished Nearest Neighbor Search
Sun Dec 28 16:35:22 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Sun Dec 28 16:35:23 2025 Finished embedding
✅ UMAP complete: (44, 2)

📊 3D Projection:
🐢 Applying CPU UMAP...
   Input: (44, 4096) → 3D
UMAP(angular_rp_forest=True, metric='cosine', n_components=3, n_jobs=1, random_state=42, verbose=True)
Sun Dec 28 16:35:23 2025 Construct fuzzy simplicial set
Sun Dec 28 16:35:23 2025 Finding Nearest Neighbors
Sun Dec 28 16:35:23 2025 Finished Nearest Neighbor Search
Sun Dec 28 16:35:23 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Sun Dec 28 16:35:23 2025 Finished embedding
✅ UMAP complete: (44, 3)

✅ Problem coordinates added to dataframe


In [7]:
# Apply UMAP to answer embeddings (2D and 3D)
print("=" * 70)
print("🔮 UMAP: Answer Embeddings")
print("=" * 70)

print("\n📊 2D Projection:")
answer_coords_2d = apply_umap(answer_embeddings, n_components=2)

print("\n📊 3D Projection:")
answer_coords_3d = apply_umap(answer_embeddings, n_components=3)

# Add to dataframe
df['ans_x2d'] = answer_coords_2d[:, 0]
df['ans_y2d'] = answer_coords_2d[:, 1]
df['ans_x3d'] = answer_coords_3d[:, 0]
df['ans_y3d'] = answer_coords_3d[:, 1]
df['ans_z3d'] = answer_coords_3d[:, 2]

print("\n✅ Answer coordinates added to dataframe")


🔮 UMAP: Answer Embeddings

📊 2D Projection:
🐢 Applying CPU UMAP...
   Input: (44, 4096) → 2D
UMAP(angular_rp_forest=True, metric='cosine', n_jobs=1, random_state=42, verbose=True)
Sun Dec 28 16:35:23 2025 Construct fuzzy simplicial set
Sun Dec 28 16:35:23 2025 Finding Nearest Neighbors
Sun Dec 28 16:35:23 2025 Finished Nearest Neighbor Search
Sun Dec 28 16:35:23 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Sun Dec 28 16:35:23 2025 Finished embedding
✅ UMAP complete: (44, 2)

📊 3D Projection:
🐢 Applying CPU UMAP...
   Input: (44, 4096) → 3D
UMAP(angular_rp_forest=True, metric='cosine', n_components=3, n_jobs=1, random_state=42, verbose=True)
Sun Dec 28 16:35:23 2025 Construct fuzzy simplicial set
Sun Dec 28 16:35:23 2025 Finding Nearest Neighbors
Sun Dec 28 16:35:23 2025 Finished Nearest Neighbor Search
Sun Dec 28 16:35:23 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Sun Dec 28 16:35:23 2025 Finished embedding
✅ UMAP complete: (44, 3)

✅ Answer coordinates added to dataframe


## 📊 Visualization Functions


In [8]:
def calculate_adaptive_opacity(n_points: int, min_opacity: float = 0.6, max_opacity: float = 0.9) -> float:
    """Calculate adaptive opacity based on number of points."""
    if n_points <= 50:
        return max_opacity
    elif n_points >= 500:
        return min_opacity
    else:
        ratio = (n_points - 50) / 450
        return max_opacity - (max_opacity - min_opacity) * ratio


def calculate_adaptive_marker_size(n_points: int, min_size: float = 4, max_size: float = 12) -> float:
    """Calculate adaptive marker size based on number of points."""
    if n_points <= 50:
        return max_size
    elif n_points >= 500:
        return min_size
    else:
        ratio = (n_points - 50) / 450
        return max_size - (max_size - min_size) * ratio


def create_2d_scatter(
    data: pd.DataFrame,
    x_col: str, y_col: str,
    color_col: str,
    color_map: Dict,
    title: str,
    hover_cols: list = None
) -> go.Figure:
    """Create interactive 2D scatter plot with LaTeX support."""
    n_points = len(data)
    opacity = calculate_adaptive_opacity(n_points)
    marker_size = calculate_adaptive_marker_size(n_points)
    
    print(f"   📊 2D plot: {n_points} points → opacity={opacity:.2f}, size={marker_size:.1f}")
    
    fig = px.scatter(
        data, x=x_col, y=y_col,
        color=color_col,
        color_discrete_map=color_map,
        hover_data=hover_cols or ['problem_preview', 'answer', 'source'],
        title=title,
        template='plotly_white'
    )
    
    fig.update_traces(marker=dict(
        size=marker_size, 
        opacity=opacity, 
        line=dict[str, float | str](width=0.5, color='#333'))
    )
    fig.update_layout(
        width=1000, height=700,
        title_x=0.5,
        title_font=dict(size=18, color='#2c3e50'),
        xaxis_title='UMAP 1', yaxis_title='UMAP 2',
        xaxis=dict(gridcolor='#e0e0e0', zerolinecolor='#ccc'),
        yaxis=dict(gridcolor='#e0e0e0', zerolinecolor='#ccc'),
        paper_bgcolor='white',
        plot_bgcolor='#fafafa',
        legend=dict(
            bgcolor='rgba(255, 255, 255, 0.95)', 
            bordercolor='#ddd', 
            borderwidth=1, 
            font=dict(color='#333')
        )
    )
    return fig


def create_3d_scatter(
    data: pd.DataFrame,
    x_col: str, y_col: str, z_col: str,
    color_col: str,
    color_map: Dict,
    title: str,
    hover_cols: list = None
) -> go.Figure:
    """Create interactive 3D scatter plot with LaTeX support."""
    n_points = len(data)
    opacity = calculate_adaptive_opacity(n_points, 0.5, 0.85)
    marker_size = calculate_adaptive_marker_size(n_points, 3, 10)
    
    print(f"   📊 3D plot: {n_points} points → opacity={opacity:.2f}, size={marker_size:.1f}")
    
    fig = px.scatter_3d(
        data, x=x_col, y=y_col, z=z_col,
        color=color_col,
        color_discrete_map=color_map,
        hover_data=hover_cols or ['problem_preview', 'answer', 'source'],
        title=title,
        template='plotly_white'
    )
    
    fig.update_traces(marker=dict(size=marker_size, opacity=opacity, line=dict[str, float | str](width=0.3, color='#333')))
    fig.update_layout(
        width=1000, height=700,
        title_x=0.5,
        title_font=dict(size=18, color='#2c3e50'),
        paper_bgcolor='white',
        scene=dict(
            xaxis_title='UMAP 1', yaxis_title='UMAP 2', zaxis_title='UMAP 3',
            bgcolor='#f8f9fa',
            xaxis=dict(backgroundcolor='#f0f0f0', gridcolor='#ddd', showbackground=True),
            yaxis=dict(backgroundcolor='#f0f0f0', gridcolor='#ddd', showbackground=True),
            zaxis=dict(backgroundcolor='#f0f0f0', gridcolor='#ddd', showbackground=True)
        ),
        legend=dict(bgcolor='rgba(255, 255, 255, 0.95)', bordercolor='#ddd', borderwidth=1, font=dict(color='#333'))
    )
    return fig

print("✅ Visualization functions defined (white theme with LaTeX support)")


✅ Visualization functions defined (white theme with LaTeX support)


## 🎨 Problem Embeddings by AIMO Version


In [9]:
# 2D: Problems by AIMO Version
print("🎨 Creating 2D visualization: Problems by AIMO Version")
fig_prob_version_2d = create_2d_scatter(
    df, 'prob_x2d', 'prob_y2d',
    'aimo_version', AIMO_COLORS,
    '🧮 AIMO Problems - 2D UMAP by Competition'
)
fig_prob_version_2d.write_html(OUTPUT_DIR / "problems_by_version_2d.html", include_mathjax='cdn')
print(f"   ✅ Saved: {OUTPUT_DIR / 'problems_by_version_2d.html'}")
# fig_prob_version_2d.show()


🎨 Creating 2D visualization: Problems by AIMO Version
   📊 2D plot: 44 points → opacity=0.90, size=12.0
   ✅ Saved: visualizations/problems_by_version_2d.html


In [10]:
# 3D: Problems by AIMO Version
print("🎨 Creating 3D visualization: Problems by AIMO Version")
fig_prob_version_3d = create_3d_scatter(
    df, 'prob_x3d', 'prob_y3d', 'prob_z3d',
    'aimo_version', AIMO_COLORS,
    '🧮 AIMO Problems - 3D UMAP by Competition'
)
fig_prob_version_3d.write_html(OUTPUT_DIR / "problems_by_version_3d.html", include_mathjax='cdn')
print(f"   ✅ Saved: {OUTPUT_DIR / 'problems_by_version_3d.html'}")
# fig_prob_version_3d.show()


🎨 Creating 3D visualization: Problems by AIMO Version
   📊 3D plot: 44 points → opacity=0.85, size=10.0
   ✅ Saved: visualizations/problems_by_version_3d.html


## 🎨 Problem Embeddings by Answer Status


In [11]:
# 2D: Problems by Answer Status
print("🎨 Creating 2D visualization: Problems by Answer Status")
fig_prob_answer_2d = create_2d_scatter(
    df, 'prob_x2d', 'prob_y2d',
    'answer_status', ANSWER_STATUS_COLORS,
    '🧮 AIMO Problems - 2D UMAP by Answer Status'
)
fig_prob_answer_2d.write_html(OUTPUT_DIR / "problems_by_answer_2d.html", include_mathjax='cdn')
print(f"   ✅ Saved: {OUTPUT_DIR / 'problems_by_answer_2d.html'}")
# fig_prob_answer_2d.show()


🎨 Creating 2D visualization: Problems by Answer Status
   📊 2D plot: 44 points → opacity=0.90, size=12.0
   ✅ Saved: visualizations/problems_by_answer_2d.html


In [12]:
# 3D: Problems by Answer Status
print("🎨 Creating 3D visualization: Problems by Answer Status")
fig_prob_answer_3d = create_3d_scatter(
    df, 'prob_x3d', 'prob_y3d', 'prob_z3d',
    'answer_status', ANSWER_STATUS_COLORS,
    '🧮 AIMO Problems - 3D UMAP by Answer Status'
)
fig_prob_answer_3d.write_html(OUTPUT_DIR / "problems_by_answer_3d.html", include_mathjax='cdn')
print(f"   ✅ Saved: {OUTPUT_DIR / 'problems_by_answer_3d.html'}")
# fig_prob_answer_3d.show()


🎨 Creating 3D visualization: Problems by Answer Status
   📊 3D plot: 44 points → opacity=0.85, size=10.0
   ✅ Saved: visualizations/problems_by_answer_3d.html


## 🎨 Answer Embeddings (Answered Problems Only)


In [13]:
# Filter to answered problems only
answered_df = df[df['has_answer'] == True].copy()
print(f"📊 Answered problems: {len(answered_df)}")

# 2D: Answers by AIMO Version
print("\n🎨 Creating 2D visualization: Answers by AIMO Version")
fig_ans_2d = create_2d_scatter(
    answered_df, 'ans_x2d', 'ans_y2d',
    'aimo_version', AIMO_COLORS,
    '🧮 AIMO Answers - 2D UMAP by Competition'
)
fig_ans_2d.write_html(OUTPUT_DIR / "answers_by_version_2d.html", include_mathjax='cdn')
print(f"   ✅ Saved: {OUTPUT_DIR / 'answers_by_version_2d.html'}")
# fig_ans_2d.show()


📊 Answered problems: 30

🎨 Creating 2D visualization: Answers by AIMO Version
   📊 2D plot: 30 points → opacity=0.90, size=12.0
   ✅ Saved: visualizations/answers_by_version_2d.html


In [14]:
# 3D: Answers by AIMO Version
print("🎨 Creating 3D visualization: Answers by AIMO Version")
fig_ans_3d = create_3d_scatter(
    answered_df, 'ans_x3d', 'ans_y3d', 'ans_z3d',
    'aimo_version', AIMO_COLORS,
    '🧮 AIMO Answers - 3D UMAP by Competition'
)
fig_ans_3d.write_html(OUTPUT_DIR / "answers_by_version_3d.html", include_mathjax='cdn')
print(f"   ✅ Saved: {OUTPUT_DIR / 'answers_by_version_3d.html'}")
# fig_ans_3d.show()


🎨 Creating 3D visualization: Answers by AIMO Version
   📊 3D plot: 30 points → opacity=0.85, size=10.0
   ✅ Saved: visualizations/answers_by_version_3d.html


## 🔗 Combined Problem + Answer Space (Joint UMAP)

Visualizes problems and their answers in the same embedding space with:
- **Color-coded by AIMO version**: AIMO 1 (red), AIMO 2 (blue), AIMO 3 (purple)
- **Problems**: Solid colors | **Answers**: Lighter shades | **Unanswered**: Very light/pastel
- **Interactive arrows**: Click legend to show/hide arrows for each AIMO competition
- **Arrows connect**: Each answered problem → its corresponding answer


In [15]:
# Create combined embeddings for joint UMAP (ALL problems + answers for answered ones)
answered_mask = df['has_answer'].values
unanswered_mask = ~answered_mask

# All problems
all_problems = problem_embeddings
# Only answered answers
answered_answers = answer_embeddings[answered_mask]

# Stack: all problems first, then answered answers
combined_embeddings = np.vstack([all_problems, answered_answers])
n_all_problems = len(all_problems)
n_answered = answered_mask.sum()
n_unanswered = unanswered_mask.sum()

print(f"📊 Combined embeddings: {combined_embeddings.shape}")
print(f"   All problems: {n_all_problems} (answered: {n_answered}, unanswered: {n_unanswered})")
print(f"   Answers: {n_answered}")

# Apply joint UMAP (2D and 3D)
print("\n🔮 Joint UMAP: Combined Problem + Answer Embeddings")
print("\n📊 2D Projection:")
combined_coords_2d = apply_umap(combined_embeddings, n_components=2)

print("\n📊 3D Projection:")
combined_coords_3d = apply_umap(combined_embeddings, n_components=3)

# Split coordinates back
prob_comb_2d = combined_coords_2d[:n_all_problems]
ans_comb_2d = combined_coords_2d[n_all_problems:]
prob_comb_3d = combined_coords_3d[:n_all_problems]
ans_comb_3d = combined_coords_3d[n_all_problems:]

# Create labels based on AIMO version and type
def create_label(row, item_type):
    """Create label like 'AIMO 1 Problem', 'AIMO 2 Answer', 'AIMO 1 Unanswered'"""
    if item_type == 'problem':
        if row['has_answer']:
            return f"{row['aimo_version']} Problem"
        else:
            return f"{row['aimo_version']} Unanswered"
    else:  # answer
        return f"{row['aimo_version']} Answer"

# Create dataframe for problems (all) - content already has line breaks from problem_preview
problems_df = pd.DataFrame({
    'x2d': prob_comb_2d[:, 0], 'y2d': prob_comb_2d[:, 1],
    'x3d': prob_comb_3d[:, 0], 'y3d': prob_comb_3d[:, 1], 'z3d': prob_comb_3d[:, 2],
    'content': df['problem_preview'].values,  # Already wrapped with <br> tags
    'answer': df['answer'].values,
    'aimo_version': df['aimo_version'].values,
    'source': df['source'].values,
    'has_answer': df['has_answer'].values,
    'is_problem': True
})
problems_df['label'] = problems_df.apply(lambda r: create_label(r, 'problem'), axis=1)

# Create dataframe for answers (only answered) - wrap answer text for hover
answered_meta = df[df['has_answer'] == True].reset_index(drop=True)
# Wrap answer content with line breaks
answer_content_wrapped = answered_meta['answer'].astype(str).apply(
    lambda x: wrap_text_for_hover(x, max_chars=70, max_lines=4)
)
answers_df = pd.DataFrame({
    'x2d': ans_comb_2d[:, 0], 'y2d': ans_comb_2d[:, 1],
    'x3d': ans_comb_3d[:, 0], 'y3d': ans_comb_3d[:, 1], 'z3d': ans_comb_3d[:, 2],
    'content': answer_content_wrapped.values,  # Wrapped with <br> tags
    'answer': answered_meta['answer'].values,
    'aimo_version': answered_meta['aimo_version'].values,
    'source': answered_meta['source'].values,
    'has_answer': True,
    'is_problem': False
})
answers_df['label'] = answers_df.apply(lambda r: create_label(r, 'answer'), axis=1)

# Combine
combined_df = pd.concat([problems_df, answers_df], ignore_index=True)

# Store indices for arrow drawing (mapping answered problems to their answers)
answered_problem_indices = problems_df[problems_df['has_answer'] == True].index.tolist()
answer_indices = list(range(len(problems_df), len(problems_df) + len(answers_df)))

print(f"\n✅ Combined dataframe: {len(combined_df)} points")
print(f"   Labels: {combined_df['label'].value_counts().to_dict()}")


📊 Combined embeddings: (74, 4096)
   All problems: 44 (answered: 30, unanswered: 14)
   Answers: 30

🔮 Joint UMAP: Combined Problem + Answer Embeddings

📊 2D Projection:
🐢 Applying CPU UMAP...
   Input: (74, 4096) → 2D
UMAP(angular_rp_forest=True, metric='cosine', n_jobs=1, random_state=42, verbose=True)
Sun Dec 28 16:35:26 2025 Construct fuzzy simplicial set
Sun Dec 28 16:35:26 2025 Finding Nearest Neighbors
Sun Dec 28 16:35:26 2025 Finished Nearest Neighbor Search
Sun Dec 28 16:35:26 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Sun Dec 28 16:35:26 2025 Finished embedding
✅ UMAP complete: (74, 2)

📊 3D Projection:
🐢 Applying CPU UMAP...
   Input: (74, 4096) → 3D
UMAP(angular_rp_forest=True, metric='cosine', n_components=3, n_jobs=1, random_state=42, verbose=True)
Sun Dec 28 16:35:26 2025 Construct fuzzy simplicial set
Sun Dec 28 16:35:26 2025 Finding Nearest Neighbors
Sun Dec 28 16:35:26 2025 Finished Nearest Neighbor Search
Sun Dec 28 16:35:26 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Sun Dec 28 16:35:26 2025 Finished embedding
✅ UMAP complete: (74, 3)

✅ Combined dataframe: 74 points
   Labels: {'AIMO 1 Problem': 10, 'AIMO 2 Problem': 10, 'AIMO 3 Problem': 10, 'AIMO 1 Answer': 10, 'AIMO 2 Answer': 10, 'AIMO 3 Answer': 10, 'AIMO 3 Unanswered': 8, 'AIMO 1 Unanswered': 3, 'AIMO 2 Unanswered': 3}


In [16]:
# 2D: Combined Problem + Answer with Arrows (labeled by AIMO version)
print("🎨 Creating 2D visualization: Combined Problem + Answer Space with Arrows")

# Color scheme for AIMO version labels (problems solid, answers lighter, unanswered dashed pattern)
COMBINED_COLORS = {
    'AIMO 1 Problem': '#E74C3C',      # Red (problem)
    'AIMO 1 Answer': '#F1948A',       # Light red (answer)
    'AIMO 1 Unanswered': '#FADBD8',   # Very light red (unanswered)
    'AIMO 2 Problem': '#3498DB',      # Blue (problem)
    'AIMO 2 Answer': '#85C1E9',       # Light blue (answer)
    'AIMO 2 Unanswered': '#D4E6F1',   # Very light blue (unanswered)
    'AIMO 3 Problem': '#9B59B6',      # Purple (problem)
    'AIMO 3 Answer': '#D7BDE2',       # Light purple (answer)
    'AIMO 3 Unanswered': '#E8DAEF',   # Very light purple (unanswered)
}

# Arrow colors for each AIMO version
ARROW_COLORS = {
    'AIMO 1': '#C0392B',  # Dark red
    'AIMO 2': '#2980B9',  # Dark blue
    'AIMO 3': '#8E44AD',  # Dark purple
}

fig_combined_2d = create_2d_scatter(
    combined_df, 'x2d', 'y2d',
    'label', COMBINED_COLORS,
    '🧮 Combined Problem & Answer Space - 2D UMAP (by AIMO Version)',
    hover_cols=['content', 'aimo_version', 'source']
)

# Add arrows from answered problems to their answers - grouped by AIMO version for interactivity
answered_problems_2d = combined_df[(combined_df['is_problem'] == True) & (combined_df['has_answer'] == True)].reset_index(drop=True)
answers_2d = combined_df[combined_df['is_problem'] == False].reset_index(drop=True)

# Group arrows by AIMO version
for aimo_ver in ['AIMO 1', 'AIMO 2', 'AIMO 3']:
    # Get indices for this AIMO version
    mask = answered_problems_2d['aimo_version'] == aimo_ver
    indices = answered_problems_2d[mask].index.tolist()
    
    if len(indices) == 0:
        continue
    
    # Collect all line segments for this AIMO version
    x_lines = []
    y_lines = []
    for i in indices:
        x_lines.extend([answered_problems_2d.loc[i, 'x2d'], answers_2d.loc[i, 'x2d'], None])
        y_lines.extend([answered_problems_2d.loc[i, 'y2d'], answers_2d.loc[i, 'y2d'], None])
    
    # Add as single trace with legend entry
    fig_combined_2d.add_trace(go.Scatter(
        x=x_lines,
        y=y_lines,
        mode='lines',
        line=dict(color=ARROW_COLORS[aimo_ver], width=1.5, dash='solid'),
        name=f'{aimo_ver} Arrows',
        legendgroup=f'{aimo_ver}_arrows',
        showlegend=True,
        hoverinfo='skip',
        opacity=0.6
    ))
    
    # Add arrowhead markers at answer positions
    arrow_x = [answers_2d.loc[i, 'x2d'] for i in indices]
    arrow_y = [answers_2d.loc[i, 'y2d'] for i in indices]
    
    fig_combined_2d.add_trace(go.Scatter(
        x=arrow_x,
        y=arrow_y,
        mode='markers',
        marker=dict(
            symbol='triangle-up',
            size=8,
            color=ARROW_COLORS[aimo_ver],
            opacity=0.7
        ),
        name=f'{aimo_ver} Arrow Tips',
        legendgroup=f'{aimo_ver}_arrows',
        showlegend=False,
        hoverinfo='skip'
    ))

n_arrows = len(answered_problems_2d)
print(f"   ➡️  Added {n_arrows} arrows (interactive by AIMO version)")
print(f"   ⚪ Unanswered problems distinguished by competition")
fig_combined_2d.write_html(OUTPUT_DIR / "combined_space_2d.html", include_mathjax='cdn')
print(f"   ✅ Saved: {OUTPUT_DIR / 'combined_space_2d.html'}")
fig_combined_2d.show()


🎨 Creating 2D visualization: Combined Problem + Answer Space with Arrows
   📊 2D plot: 74 points → opacity=0.88, size=11.6
   ➡️  Added 30 arrows (interactive by AIMO version)
   ⚪ Unanswered problems distinguished by competition
   ✅ Saved: visualizations/combined_space_2d.html


In [17]:
# 3D: Combined Problem + Answer with Arrows (labeled by AIMO version)
print("🎨 Creating 3D visualization: Combined Problem + Answer Space with Arrows")

# Use same color scheme as 2D
fig_combined_3d = create_3d_scatter(
    combined_df, 'x3d', 'y3d', 'z3d',
    'label', COMBINED_COLORS,
    '🧮 Combined Problem & Answer Space - 3D UMAP (by AIMO Version)',
    hover_cols=['content', 'aimo_version', 'source']
)

# Add arrows from answered problems to their answers in 3D - grouped by AIMO version
answered_problems_3d = combined_df[(combined_df['is_problem'] == True) & (combined_df['has_answer'] == True)].reset_index(drop=True)
answers_3d = combined_df[combined_df['is_problem'] == False].reset_index(drop=True)

# Group arrows by AIMO version for interactivity
for aimo_ver in ['AIMO 1', 'AIMO 2', 'AIMO 3']:
    # Get indices for this AIMO version
    mask = answered_problems_3d['aimo_version'] == aimo_ver
    indices = answered_problems_3d[mask].index.tolist()
    
    if len(indices) == 0:
        continue
    
    # Collect all line segments for this AIMO version
    x_lines = []
    y_lines = []
    z_lines = []
    for i in indices:
        x_lines.extend([answered_problems_3d.loc[i, 'x3d'], answers_3d.loc[i, 'x3d'], None])
        y_lines.extend([answered_problems_3d.loc[i, 'y3d'], answers_3d.loc[i, 'y3d'], None])
        z_lines.extend([answered_problems_3d.loc[i, 'z3d'], answers_3d.loc[i, 'z3d'], None])
    
    # Add as single trace with legend entry
    fig_combined_3d.add_trace(go.Scatter3d(
        x=x_lines,
        y=y_lines,
        z=z_lines,
        mode='lines',
        line=dict(color=ARROW_COLORS[aimo_ver], width=3),
        name=f'{aimo_ver} Arrows',
        legendgroup=f'{aimo_ver}_arrows',
        showlegend=True,
        hoverinfo='skip',
        opacity=0.6
    ))
    
    # Calculate cone directions for arrowheads
    u_vals = []
    v_vals = []
    w_vals = []
    x_vals = []
    y_vals = []
    z_vals = []
    
    for i in indices:
        dx = answers_3d.loc[i, 'x3d'] - answered_problems_3d.loc[i, 'x3d']
        dy = answers_3d.loc[i, 'y3d'] - answered_problems_3d.loc[i, 'y3d']
        dz = answers_3d.loc[i, 'z3d'] - answered_problems_3d.loc[i, 'z3d']
        
        # Normalize direction
        length = np.sqrt(dx**2 + dy**2 + dz**2)
        if length > 0:
            u_vals.append(dx / length)
            v_vals.append(dy / length)
            w_vals.append(dz / length)
            x_vals.append(answers_3d.loc[i, 'x3d'])
            y_vals.append(answers_3d.loc[i, 'y3d'])
            z_vals.append(answers_3d.loc[i, 'z3d'])
    
    # Add cone markers for arrowheads (same legend group as lines)
    if len(x_vals) > 0:
        fig_combined_3d.add_trace(go.Cone(
            x=x_vals, y=y_vals, z=z_vals,
            u=u_vals, v=v_vals, w=w_vals,
            sizemode='absolute',
            sizeref=0.4,
            anchor='tip',
            showscale=False,
            colorscale=[[0, ARROW_COLORS[aimo_ver]], [1, ARROW_COLORS[aimo_ver]]],
            name=f'{aimo_ver} Arrow Tips',
            legendgroup=f'{aimo_ver}_arrows',
            showlegend=False,
            hoverinfo='skip',
            opacity=0.7
        ))

n_arrows = len(answered_problems_3d)
print(f"   ➡️  Added {n_arrows} arrows (interactive by AIMO version)")
print(f"   ⚪ Unanswered problems distinguished by competition")
fig_combined_3d.write_html(OUTPUT_DIR / "combined_space_3d.html", include_mathjax='cdn')
print(f"   ✅ Saved: {OUTPUT_DIR / 'combined_space_3d.html'}")
fig_combined_3d.show()


🎨 Creating 3D visualization: Combined Problem + Answer Space with Arrows
   📊 3D plot: 74 points → opacity=0.83, size=9.6
   ➡️  Added 30 arrows (interactive by AIMO version)
   ⚪ Unanswered problems distinguished by competition
   ✅ Saved: visualizations/combined_space_3d.html


## 📊 Summary


In [18]:
print("=" * 70)
print("📊 AIMO EMBEDDING VISUALIZATION SUMMARY")
print("=" * 70)

print(f"\n🖥️  Backend: {'GPU (RAPIDS cuML)' if RAPIDS_AVAILABLE else 'CPU (umap-learn)'}")

print(f"\n📁 Data:")
print(f"   Total problems: {len(df)}")
print(f"   Embedding dimension: {problem_embeddings.shape[1]}")
print(f"   Problems with answers: {df['has_answer'].sum()}")
print(f"   Unanswered problems: {(~df['has_answer']).sum()}")

print(f"\n📈 By Competition:")
for ver in ['AIMO 1', 'AIMO 2', 'AIMO 3']:
    count = (df['aimo_version'] == ver).sum()
    answered = ((df['aimo_version'] == ver) & (df['has_answer'] == True)).sum()
    print(f"   {ver}: {count} problems ({answered} with answers)")

print(f"\n📁 Output files saved to: {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.glob("*.html")):
    print(f"   • {f.name}")

print("\n" + "=" * 70)
print("✅ Visualization complete!")
print("   Open the HTML files in a browser for interactive exploration.")
print("=" * 70)


📊 AIMO EMBEDDING VISUALIZATION SUMMARY

🖥️  Backend: CPU (umap-learn)

📁 Data:
   Total problems: 44
   Embedding dimension: 4096
   Problems with answers: 30
   Unanswered problems: 14

📈 By Competition:
   AIMO 1: 13 problems (10 with answers)
   AIMO 2: 13 problems (10 with answers)
   AIMO 3: 18 problems (10 with answers)

📁 Output files saved to: visualizations
   • answers_by_version_2d.html
   • answers_by_version_3d.html
   • combined_space_2d.html
   • combined_space_3d.html
   • problems_by_answer_2d.html
   • problems_by_answer_3d.html
   • problems_by_version_2d.html
   • problems_by_version_3d.html

✅ Visualization complete!
   Open the HTML files in a browser for interactive exploration.
